In [1]:
import polars as pl
from dotenv import load_dotenv
import os

load_dotenv(".env")


True

In [ ]:
from src.pipelines.simple_time_clustering import SimpleTimeClustering
from src.repository.alarm_graph_repository import AlarmGraphRepository

lazy_frame = pl.scan_parquet("data/raw/alarm_history_dump.parquet")
graph_repo = AlarmGraphRepository("clustering_history.db")

SimpleTimeClustering.train(lazy_frame, graph_repo, threshold_minutes=5)

display(graph_repo.preview_nodes())
graph_repo.close()

Processando nós: 100%|██████████| 912/912 [00:26<00:00, 34.12nó/s] 


alert_id,incident,alert_type,start_time,end_time,node_id
str,i32,str,datetime[μs],datetime[μs],str
"""e542b2ff-667b-4a7d-bdad-49e839…",2,"""MANAGEMENT_BOARD_TEMPERATURE_R…",2026-03-25 03:30:41.939224,null,"""e4efe34f-1d6d-4fd2-ad89-6e6b8e…"
"""cd425e12-4c38-469b-b7c6-aa4ec2…",0,"""MANAGEMENT_BOARD_TEMPERATURE_R…",2026-03-08 10:41:06.585867,null,"""e4efe34f-1d6d-4fd2-ad89-6e6b8e…"
"""72232f7f-a4a3-40ba-bd5c-1bb638…",0,"""MANAGEMENT_BOARD_TEMPERATURE_R…",2026-03-12 01:56:07.298777,null,"""e4efe34f-1d6d-4fd2-ad89-6e6b8e…"
"""bde68065-2c76-4dba-b79a-fd0bf4…",2,"""MANAGEMENT_BOARD_TEMPERATURE_R…",2026-03-25 01:06:10.053400,null,"""e4efe34f-1d6d-4fd2-ad89-6e6b8e…"
"""93b9908a-392e-4167-adea-9efbe0…",0,"""MANAGEMENT_BOARD_TEMPERATURE_R…",2026-03-09 09:15:52.219245,null,"""e4efe34f-1d6d-4fd2-ad89-6e6b8e…"
…,…,…,…,…,…
"""4f2d51c4-303b-4f42-a782-30905c…",1,"""MANAGEMENT_BOARD_TEMPERATURE_R…",2026-03-16 11:25:46.436851,null,"""e4efe34f-1d6d-4fd2-ad89-6e6b8e…"
"""1e3d0b6c-082f-430e-bc93-b2cce3…",2,"""MANAGEMENT_BOARD_TEMPERATURE_R…",2026-03-27 04:15:41.228377,null,"""e4efe34f-1d6d-4fd2-ad89-6e6b8e…"
"""a4662268-5eb5-4f7e-92f0-56de0e…",0,"""MANAGEMENT_BOARD_TEMPERATURE_R…",2026-03-03 02:41:07.563711,null,"""e4efe34f-1d6d-4fd2-ad89-6e6b8e…"


In [3]:
from src.preprocess.binarize_preprocessor import BinarizePreprocessor
from src.graphing.pcmci_correlation import PCMCICorrelation
from src.repository.alarm_graph_repository import AlarmGraphRepository

graph_repo = AlarmGraphRepository("clustering_history.db")

incidents = graph_repo.get_incidents()
preprocessor = BinarizePreprocessor(bin_size_seconds=60*5)
df = preprocessor.remove_stopword_types(incidents)
df = preprocessor.filter_incidents(df)
bin_df = preprocessor.binarize(df)

stopwords removidos (1): {'MANAGEMENT_BOARD_TEMPERATURE_READING'}


In [ ]:
results = PCMCICorrelation(pc_alpha=0.05).correlate(bin_df)

In [ ]:
def parse_pcmci_results(results: dict, var_names: list[str]) -> pl.DataFrame:
    graph     = results["graph"]        # (N, N, tau_max+1)
    p_matrix  = results["p_matrix"]     # (N, N, tau_max+1)
    val_matrix = results["val_matrix"]  # (N, N, tau_max+1)

    N = len(var_names)
    rows = []

    for i in range(N):
        for j in range(N):
            if i == j:
                continue
            for tau in range(1, graph.shape[2]):  # pula lag 0
                link = graph[i, j, tau]
                if link != "":
                    rows.append({
                        "cause":     var_names[i],
                        "effect":    var_names[j],
                        "lag":       tau,
                        "link_type": link,
                        "p_value":   float(p_matrix[i, j, tau]),
                        "val":       float(val_matrix[i, j, tau]),
                    })

    return pl.DataFrame(rows)

edges_df = parse_pcmci_results(results, bin_df.var_names)
edges_df

shape: (3, 6)
┌─────────────────────────────┬────────────────────────────┬─────┬───────────┬──────────┬──────────┐
│ cause                       ┆ effect                     ┆ lag ┆ link_type ┆ p_value  ┆ val      │
│ ---                         ┆ ---                        ┆ --- ┆ ---       ┆ ---      ┆ ---      │
│ str                         ┆ str                        ┆ i64 ┆ str       ┆ f64      ┆ f64      │
╞═════════════════════════════╪════════════════════════════╪═════╪═══════════╪══════════╪══════════╡
│ THREESCALE_API_MANAGER_WORK ┆ THREESCALE_API_MANAGER_LIS ┆ 1   ┆ -->       ┆ 0.005988 ┆ 0.0065   │
│ ER_…                        ┆ TENE…                      ┆     ┆           ┆          ┆          │
│ THREESCALE_CLUSTER_OPENSHIF ┆ THREESCALE_CLUSTER_PROMETH ┆ 4   ┆ -->       ┆ 0.011976 ┆ 0.003439 │
│ T_U…                        ┆ EUS_…                      ┆     ┆           ┆          ┆          │
│ THREESCALE_CLUSTER_PROMETHE ┆ THREESCALE_CLUSTER_OPENSHI ┆ 4   ┆ -->       

In [8]:
graph_repo.close()